In [2]:
!pip install langchain langchain-google-genai pydantic pywhatkit


In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import StructuredTool
from pydantic import BaseModel
import subprocess
import os
import os
import google.generativeai as genai
from dotenv import load_dotenv

In [ ]:


# 1. Load the environment variables from .env
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")  # Replace with your key securely


In [5]:
class CommandInput(BaseModel):
    args: str = ""

def create_command_tool(command_name: str, base_command: str, description: str):
    def command_func(**kwargs) -> str:
        args = kwargs.get("args", "")
        try:
            result = subprocess.run(
                f"{base_command} {args}".strip(),
                shell=True,
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            return result.stdout or "✅ Command executed successfully"
        except subprocess.CalledProcessError as e:
            return f"❌ Error:\\n{e.stderr or e.stdout}"
    return StructuredTool.from_function(
        func=command_func,
        name=command_name,
        description=description,
        args_schema=CommandInput
    )


In [6]:
tools = [
    create_command_tool("date_command", "date", "Show current system date and time."),
    create_command_tool("cal_command", "cal", "Display calendar."),
    create_command_tool("ifconfig_command", "ifconfig", "Show network interface configuration."),
    create_command_tool("ls_command", "ls", "List directory contents. Optional args like -l or -a."),
    create_command_tool("mkdir_command", "mkdir", "Create a new directory. Provide name as argument."),
    create_command_tool("useradd_command", "sudo useradd", "Add a new user. Provide username."),
    create_command_tool("userdel_command", "sudo userdel", "Delete a user. Provide username."),
    create_command_tool("list_file_permissions", "ls -l", "List all file permissions in current directory."),
    create_command_tool("show_file_permissions", "ls -l", "Show permission of a particular file. Pass filename."),
    create_command_tool("change_file_permissions", "chmod", "Change file permissions. Example: 755 myfile.txt")
]


In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)


C:\Users\lakshya\AppData\Local\Temp\ipykernel_18520\3978272235.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [8]:
# ✅ Additional tools: Commands 11 to 25
more_tools = [
    create_command_tool("create_file", "touch", "Create a new file. Provide filename."),
    create_command_tool("change_directory", "cd", "Change directory. Provide path."),
    create_command_tool("switch_user", "sudo su", "Change user (superuser switch)."),
    create_command_tool("pip_install", "pip install", "Install Python library using pip."),
    create_command_tool("open_notepad", "notepad", "Open a Notepad file (Windows only). Provide filename."),
    create_command_tool("gedit_editor", "gedit", "Open a file in gedit editor (Linux GUI only). Provide filename."),
    create_command_tool("vim_editor", "vim", "Open a file in vim editor. Provide filename."),
    create_command_tool("yum_config", "yum-config-manager", "Configure yum repository or settings."),
    create_command_tool("ssh_keygen", "ssh-keygen -t rsa", "Generate an SSH key using RSA algorithm."),
    create_command_tool("scp_transfer", "scp", "Transfer files using SCP. Format: source destination"),
    create_command_tool("minikube_requirements", "echo", "Show Minikube requirements or add to path."),
    create_command_tool("minikube_start", "minikube start", "Start Minikube cluster."),
    create_command_tool("install_httpd", "yum install httpd", "Install Apache HTTP server."),
    create_command_tool("check_httpd", "rpm -q httpd", "Check if Apache HTTPD is installed."),
    create_command_tool("auto_install_httpd", "yum install httpd -y", "Auto install Apache HTTPD server with confirmation.")
]

# Add them to main tools list
tools.extend(more_tools)


In [11]:
user_query = input("💬 Ask the AI to run a command (e.g., Create directory test): ")
result = agent.invoke(user_query)

print("🧠 AI Response:", result)


💬 Ask the AI to run a command (e.g., Create directory test):  list all




> Entering new AgentExecutor chain...

Invoking: `ls_command` with `{'args': ''}`


Linux_AI_Agent_Tools (2).ipynb
Untitled.ipynb
agents-streamlit.py
api.env
get_calendar.py
get_date.py
lakshya123
requirements.txt
test.py
tool_03.py
tool_04.py
tool_05.py
tool_06.py
tool_07.py
tool_08.py
tool_09.py
tool_10.py
tool_11.py
tool_12.py
tool_13.py
tool_14.py
tool_15.py
tool_16.py
tool_17.py
tool_18.py
tool_19.py
tool_20.py
tool_21.py
tool_22.py
tool_23.py
tool_24.py
tool_25.py
tool_26.py
tool_27.py
tool_28.py
tool_29.py
tool_30.py
tool_31.py
tool_32.py
tool_33.py
tool_34.py
tool_35.py
tool_36.py
tool_37.py
tool_38.py
tool_39.py
tool_40.py
tool_41.py
tool_42.py
tool_43.py
tool_44.py
tool_45.py
tool_46.py
tool_47.py
tool_48.py
tool_49.py
tool_50.py
tool_51.py


ChatGoogleGenerativeAIError: Invalid argument provided to Gemini: 400 * GenerateContentRequest.contents[2].parts: contents.parts must not be empty.


In [28]:
from langchain.tools import StructuredTool
from pydantic import BaseModel
import subprocess

# ✅ Define input schema (even if no args needed)
class DateInput(BaseModel):
    args: str = ""  # Optional argument if needed later

# ✅ Define the function with docstring
def run_date_command(args: str = "") -> str:
    """
    Show the current system date and time using the `date` command.
    """
    try:
        result = subprocess.run(
            f"date {args}".strip(),
            shell=True,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        return f"❌ Error:\n{e.stderr or e.stdout}"

# ✅ Wrap it as a StructuredTool
date_command_tool = StructuredTool.from_function(
    func=run_date_command,
    name="run_date_command",
    description="Show the current system date and time. Optionally accepts extra arguments.",
    args_schema=DateInput
)


In [29]:
print(date_command_tool.run({"args": ""}))


❌ Error:
The current date is: Thu 07/17/2025 
Enter the new date: (mm-dd-yy) 


In [31]:
pip install paramiko


Note: you may need to restart the kernel to use updated packages.


In [32]:
import paramiko

def run_ssh_command(host, username, password, command) -> str:
    """
    Connect to a remote Linux machine via SSH and run the given command.
    """
    try:
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username=username, password=password)

        stdin, stdout, stderr = ssh.exec_command(command)
        output = stdout.read().decode().strip()
        error = stderr.read().decode().strip()

        ssh.close()

        if error:
            return f"❌ Error:\n{error}"
        return output

    except Exception as e:
        return f"❌ SSH connection failed: {str(e)}"


In [33]:
from langchain.tools import StructuredTool
from pydantic import BaseModel

class SSHDateInput(BaseModel):
    host: str
    username: str
    password: str
    args: str = ""

def ssh_date_command(host: str, username: str, password: str, args: str) -> str:
    """
    Show date/time from remote Linux system.
    """
    return run_ssh_command(host, username, password, f"date {args}")

ssh_date_command_tool = StructuredTool.from_function(
    func=ssh_date_command,
    name="ssh_date_command",
    description="Show date/time from remote Linux system.",
    args_schema=SSHDateInput
)


In [34]:
class SSHCalInput(BaseModel):
    host: str
    username: str
    password: str
    args: str = ""

def ssh_cal_command(host: str, username: str, password: str, args: str) -> str:
    """
    Show calendar on a remote Linux system.
    """
    return run_ssh_command(host, username, password, f"cal {args}")

ssh_cal_command_tool = StructuredTool.from_function(
    func=ssh_cal_command,
    name="ssh_cal_command",
    description="Show calendar on a remote Linux system.",
    args_schema=SSHCalInput
)


In [35]:
class SSHIfconfigInput(BaseModel):
    host: str
    username: str
    password: str
    args: str = ""

def ssh_ifconfig_command(host: str, username: str, password: str, args: str) -> str:
    """
    Show network configuration.
    """
    return run_ssh_command(host, username, password, f"ifconfig {args}")

ssh_ifconfig_command_tool = StructuredTool.from_function(
    func=ssh_ifconfig_command,
    name="ssh_ifconfig_command",
    description="Show network configuration.",
    args_schema=SSHIfconfigInput
)


In [36]:
class SSHListFilesInput(BaseModel):
    host: str
    username: str
    password: str
    args: str = ""

def ssh_ls_command(host: str, username: str, password: str, args: str) -> str:
    """
    List files/directories on remote system.
    """
    return run_ssh_command(host, username, password, f"ls {args}")

ssh_ls_command_tool = StructuredTool.from_function(
    func=ssh_ls_command,
    name="ssh_ls_command",
    description="List files/directories on remote system.",
    args_schema=SSHListFilesInput
)


In [37]:
class SSHMkdirInput(BaseModel):
    host: str
    username: str
    password: str
    args: str

def ssh_mkdir_command(host: str, username: str, password: str, args: str) -> str:
    """
    Create a directory on the remote system.
    """
    return run_ssh_command(host, username, password, f"mkdir {args}")

ssh_mkdir_command_tool = StructuredTool.from_function(
    func=ssh_mkdir_command,
    name="ssh_mkdir_command",
    description="Create a directory on the remote system.",
    args_schema=SSHMkdirInput
)


In [45]:
output = ssh_ls_command(
    host="192.168.206.243",
    username="root",
    password="redhat",
    args=""
)
print(output)


a
anaconda-ks.cfg
backup_project
data1
Desktop
directory
docker_compose_file
Dockerfile
Documents
Downloads
jenkins
lakshya-test
make
multi_process
Music
named
Pictures
Public
Templates
Videos
vlc-docker
